# Introduction to Molecular Dynamics

Molecular dynamics, often abbreviated MD, is a computational method for studying how atoms and molecules move over time. In an MD simulation, we choose a model for the forces between particles, then repeatedly update each particle's position and velocity using Newton's laws of motion.

This notebook introduces the core ideas behind a simple MD simulation:

1. Represent particles with positions, velocities, and masses.
2. Compute forces from a potential energy function.
3. Advance the system through time with a numerical integrator.
4. Inspect the resulting trajectory and energy behavior.


## 1. The Basic MD Loop

At a high level, molecular dynamics follows this cycle:

1. Start from initial positions and velocities.
2. Use a force field to compute forces.
3. Update positions and velocities over a small time step.
4. Repeat many times to build a trajectory.

The time step must be small enough to resolve fast particle motion. In real atomistic simulations, a common time step is around 1 femtosecond, or `1e-15` seconds.

## 2. A Simple Pair Potential

For this introduction, we will model two particles interacting through the Lennard-Jones potential:

$$U(r) = 4\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6 \right]$$

Here:

- `r` is the distance between the two particles.
- `epsilon` controls the depth of the attractive well.
- `sigma` sets the approximate particle size.

The short-range repulsive term prevents particles from collapsing into each other. The longer-range attractive term gives a stable preferred separation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
def lennard_jones_potential(r, epsilon=1.0, sigma=1.0):
    """Return the Lennard-Jones potential energy for separation r."""
    sr6 = (sigma / r) ** 6
    return 4 * epsilon * (sr6 ** 2 - sr6)


def lennard_jones_force_magnitude(r, epsilon=1.0, sigma=1.0):
    """Return force magnitude along the separation vector."""
    sr6 = (sigma / r) ** 6
    return 24 * epsilon * (2 * sr6 ** 2 - sr6) / r

In [ ]:
r = np.linspace(0.85, 3.0, 500)
u = lennard_jones_potential(r)

plt.plot(r, u)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Particle separation, r")
plt.ylabel("Potential energy, U(r)")
plt.title("Lennard-Jones Potential")
plt.ylim(-1.2, 3.0)
plt.show()

## 3. Velocity Verlet Integration

A molecular dynamics integrator updates positions and velocities from one time step to the next. A common choice is the Velocity Verlet algorithm:

$$x(t + \Delta t) = x(t) + v(t)\Delta t + \frac{1}{2}a(t)\Delta t^2$$

$$v(t + \Delta t) = v(t) + \frac{1}{2}\left[a(t) + a(t + \Delta t)\right]\Delta t$$

This method is popular because it is simple and has good energy behavior for many MD problems.

In [ ]:
def compute_forces(positions, epsilon=1.0, sigma=1.0):
    """Compute Lennard-Jones forces for a two-particle 2D system."""
    displacement = positions[1] - positions[0]
    distance = np.linalg.norm(displacement)
    direction = displacement / distance
    magnitude = lennard_jones_force_magnitude(distance, epsilon, sigma)

    forces = np.zeros_like(positions)
    forces[0] = -magnitude * direction
    forces[1] = magnitude * direction
    return forces


def total_energy(positions, velocities, mass=1.0):
    displacement = positions[1] - positions[0]
    distance = np.linalg.norm(displacement)
    kinetic = 0.5 * mass * np.sum(velocities ** 2)
    potential = lennard_jones_potential(distance)
    return kinetic + potential, kinetic, potential

In [ ]:
# Initial two-particle system in reduced units.
mass = 1.0
dt = 0.005
n_steps = 4000

positions = np.array([
    [-0.65, 0.0],
    [0.65, 0.0],
], dtype=float)

velocities = np.array([
    [0.0, 0.35],
    [0.0, -0.35],
], dtype=float)

trajectory = np.zeros((n_steps, 2, 2))
energies = np.zeros((n_steps, 3))
times = np.arange(n_steps) * dt

forces = compute_forces(positions)

for step in range(n_steps):
    trajectory[step] = positions
    energies[step] = total_energy(positions, velocities, mass)

    accelerations = forces / mass
    positions = positions + velocities * dt + 0.5 * accelerations * dt ** 2

    new_forces = compute_forces(positions)
    new_accelerations = new_forces / mass
    velocities = velocities + 0.5 * (accelerations + new_accelerations) * dt
    forces = new_forces

In [ ]:
plt.plot(trajectory[:, 0, 0], trajectory[:, 0, 1], label="particle 1")
plt.plot(trajectory[:, 1, 0], trajectory[:, 1, 1], label="particle 2")
plt.scatter(trajectory[0, :, 0], trajectory[0, :, 1], color="black", zorder=3, label="start")
plt.axis("equal")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Two-Particle Trajectory")
plt.legend()
plt.show()

In [ ]:
plt.plot(times, energies[:, 0], label="total")
plt.plot(times, energies[:, 1], label="kinetic")
plt.plot(times, energies[:, 2], label="potential")
plt.xlabel("Time")
plt.ylabel("Energy")
plt.title("Energy During the Simulation")
plt.legend()
plt.show()

## 4. Things to Try

Try changing the simulation parameters above and rerunning the notebook.

- Increase or decrease `dt`. What happens to total energy conservation?
- Increase `n_steps`. Does the trajectory remain stable?
- Change the starting separation between the particles.
- Change the initial velocities.

These experiments illustrate a central lesson of molecular dynamics: the force field, time step, and initial conditions all shape the resulting trajectory.

## 5. From This Toy Model to Real MD

Real molecular dynamics simulations usually include many more ingredients:

- Thousands to millions of atoms.
- Bonded terms for bonds, angles, and dihedrals.
- Nonbonded interactions such as electrostatics and van der Waals forces.
- Periodic boundary conditions.
- Thermostats and barostats to control temperature and pressure.
- Careful validation against experiments or higher-level calculations.

Even so, the core idea remains the same: compute forces, update positions and velocities, and analyze the trajectory.